Import libraries

In [2]:
import itertools
import random
from typing import List, Dict, Tuple, Optional

print("Libraries imported successfully!")
print(f"Python version: {__import__('sys').version}")

Libraries imported successfully!
Python version: 3.10.9 (tags/v3.10.9:1dd9be6, Dec  6 2022, 20:01:21) [MSC v.1934 64 bit (AMD64)]


In [3]:
# Complete BFBCandidateGenerator Class 

class BFBCandidateGenerator:
    """
    Generate all possible BFB sequences that could produce a given copy number pattern.
    
    BFB = Breakage-Fusion-Bridge cycle
    This is a simplified demonstration of the combinatorial nature of BFB sequences.
    
    """
    
    def __init__(self, copy_numbers: List[int], foldbacks: List[bool]):
        """
        Initialize with observed data.
        
        Args:
            copy_numbers: List of copy numbers (e.g., [2, 4, 6, 4, 2])
            foldbacks: List of booleans indicating foldback positions
        """
        self.copy_numbers = copy_numbers
        self.foldbacks = foldbacks
        self.candidates = []
        
        # BFB cycle types (simplified for demonstration)
        # Each cycle type applies a different transformation
        self.cycle_types = {
            'A': self._cycle_type_a,
            'B': self._cycle_type_b,
            'C': self._cycle_type_c
        }
        
        print(f" BFBCandidateGenerator initialized")
        print(f"   Pattern: {copy_numbers}")
        print(f"   Foldbacks: {foldbacks}")
        print(f"   Cycle types available: {list(self.cycle_types.keys())}")
    
    def _cycle_type_a(self, seq: List[int]) -> List[int]:
        """Cycle type A: Simple duplication. Example: [1,2] → [1,2,1,2]"""
        return seq + seq
    
    def _cycle_type_b(self, seq: List[int]) -> List[int]:
        """Cycle type B: Duplication with break in the middle. Example: [1,2,3,4] → [1,2,3,4,3,4]"""
        mid = len(seq) // 2
        return seq[:mid] + seq[mid:] * 2
    
    def _cycle_type_c(self, seq: List[int]) -> List[int]:
        """Cycle type C: Reverse duplication. Example: [1,2,3] → [1,2,3,3,2,1]"""
        return seq + seq[::-1]
    
    def simulate_bfb(self, sequence: List[str], starting: List[int] = None) -> List[int]:
        """
        Simulate the copy numbers produced by a BFB sequence.
        
        Args:
            sequence: List of cycle types (e.g., ['A', 'B', 'A'])
            starting: Initial chromosome state (default: [1, 1])
            
        Returns:
            List of copy numbers after applying all cycles
        """
        if starting is None:
            starting = [1, 1]
        
        current = starting.copy()
        
        for cycle in sequence:
            if cycle in self.cycle_types:
                current = self.cycle_types[cycle](current)
            else:
                raise ValueError(f"Unknown cycle type: {cycle}")
        
        return current
    
    def generate_sequences(self, max_cycles: int = 4) -> List[List[str]]:
        """
        Generate all possible BFB sequences up to max_cycles.
        
        Args:
            max_cycles: Maximum number of cycles to consider
            
        Returns:
            List of all possible BFB sequences
        """
        all_sequences = []
        cycle_names = list(self.cycle_types.keys())
        
        for length in range(1, max_cycles + 1):
            # Generate all combinations with replacement
            for combo in itertools.combinations_with_replacement(cycle_names, length):
                # Generate all permutations of each combo
                for perm in set(itertools.permutations(combo)):
                    all_sequences.append(list(perm))
        
        return all_sequences
    
    def score_sequence(self, sequence: List[str]) -> Dict[str, float]:
        """
        Score how well a BFB sequence explains the observed data.
        
        Args:
            sequence: BFB sequence to evaluate
            
        Returns:
            Dict with match percentage and detailed metrics
        """
        predicted = self.simulate_bfb(sequence)
        observed = self.copy_numbers
        
        # Calculate match percentage (first min(len(predicted), len(observed)) positions)
        max_len = min(len(predicted), len(observed))
        
        if max_len == 0:
            return {
                'match_percentage': 0.0,
                'matches': 0,
                'total_positions': 0,
                'avg_diff': 0.0,
                'predicted': predicted
            }
        
        matches = 0
        for i in range(max_len):
            if predicted[i] == observed[i]:
                matches += 1
        
        match_percentage = matches / max_len
        
        # Calculate copy number difference
        diff = 0
        for i in range(max_len):
            diff += abs(predicted[i] - observed[i])
        avg_diff = diff / max_len
        
        return {
            'match_percentage': match_percentage,
            'matches': matches,
            'total_positions': max_len,
            'avg_diff': avg_diff,
            'predicted': predicted
        }
    
    def run(self, max_cycles: int = 4) -> List[Dict]:
        """
        Main method: Generate and rank all BFB candidates.
        
        Args:
            max_cycles: Maximum number of cycles to consider
            
        Returns:
            Ranked list of BFB candidates with scores
        """
        print("=" * 60)
        print("BFB CANDIDATE GENERATOR")
        print("=" * 60)
        print(f"\nInput Pattern: {self.copy_numbers}")
        print(f"Foldbacks: {self.foldbacks}")
        print(f"Searching up to {max_cycles} cycles...\n")
        
        sequences = self.generate_sequences(max_cycles)
        print(f"Generated {len(sequences)} possible sequences to evaluate")
        
        results = []
        for seq in sequences:
            score = self.score_sequence(seq)
            results.append({
                'sequence': seq,
                'score': score['match_percentage'],
                'matches': score['matches'],
                'total': score['total_positions'],
                'avg_diff': score['avg_diff'],
                'predicted': score['predicted']
            })
        
        # Sort by match percentage (highest first)
        results.sort(key=lambda x: x['score'], reverse=True)
        
        self.candidates = results
        return results
    
    def print_summary(self, top_n: int = 10):
        """
        Print a summary of the top N candidates.
        
        Args:
            top_n: Number of top candidates to display
        """
        print("\n" + "=" * 60)
        print(f"TOP {top_n} CANDIDATES")
        print("=" * 60)
        
        if not self.candidates:
            print("No candidates found. Run run() first.")
            return
        
        for i, candidate in enumerate(self.candidates[:top_n]):
            sequence_str = " -> ".join(candidate['sequence'])  # Use -> instead of → for display
            match_pct = candidate['score'] * 100
            
            print(f"\n{i+1}. Sequence: {sequence_str}")
            print(f"   Match: {match_pct:.1f}% ({candidate['matches']}/{candidate['total']} positions)")
            print(f"   Predicted: {candidate['predicted']}")
            print(f"   Average Difference: {candidate['avg_diff']:.2f}")
    
    def get_best_sequences(self, threshold: float = 0.8) -> List[Dict]:
        """
        Get all sequences that meet a minimum match threshold.
        
        Args:
            threshold: Minimum match percentage (0.0 to 1.0)
            
        Returns:
            List of candidate sequences above the threshold
        """
        if not self.candidates:
            self.run()
        
        return [c for c in self.candidates if c['score'] >= threshold]
    
    def export_sequences(self, filename: str = "bfb_candidates.txt"):
        """
        Export all sequences to a text file.
        Uses ASCII-safe characters to avoid encoding issues.
        
        Args:
            filename: Output filename
        """
        if not self.candidates:
            print("No candidates found. Run run() first.")
            return
        
        # Open with UTF-8 encoding for maximum compatibility
        try:
            # Try UTF-8 first (best for most systems)
            with open(filename, 'w', encoding='utf-8') as f:
                f.write("=" * 50 + "\n")
                f.write("BFB CANDIDATE SEQUENCES\n")
                f.write("=" * 50 + "\n")
                f.write(f"Input Pattern: {self.copy_numbers}\n")
                f.write(f"Foldbacks: {self.foldbacks}\n")
                f.write(f"Total Candidates: {len(self.candidates)}\n\n")
                
                for i, c in enumerate(self.candidates):
                    sequence_str = " -> ".join(c['sequence'])  # Use -> instead of →
                    f.write(f"{i+1}. {sequence_str}\n")
                    f.write(f"   Match: {c['score']*100:.1f}%\n")
                    f.write(f"   Predicted: {c['predicted']}\n\n")
            
            print(f"Exported {len(self.candidates)} candidates to {filename} (UTF-8)")
            
        except UnicodeEncodeError:
            # Fallback: Use ASCII-only encoding
            with open(filename, 'w', encoding='ascii', errors='ignore') as f:
                f.write("=" * 50 + "\n")
                f.write("BFB CANDIDATE SEQUENCES\n")
                f.write("=" * 50 + "\n")
                f.write(f"Input Pattern: {self.copy_numbers}\n")
                f.write(f"Foldbacks: {self.foldbacks}\n")
                f.write(f"Total Candidates: {len(self.candidates)}\n\n")
                
                for i, c in enumerate(self.candidates):
                    sequence_str = " -> ".join(c['sequence'])
                    f.write(f"{i+1}. {sequence_str}\n")
                    f.write(f"   Match: {c['score']*100:.1f}%\n")
                    f.write(f"   Predicted: {c['predicted']}\n\n")
            
            print(f" Exported {len(self.candidates)} candidates to {filename} (ASCII-only)")

print(" BFBCandidateGenerator class defined successfully")

 BFBCandidateGenerator class defined successfully


In [4]:
# Demo 1 - Basic Pattern

print("\n" + "=" * 60)
print("DEMO 1: Basic Pattern")
print("=" * 60)

# Example 1: Simple pattern
copy_numbers = [2, 4, 6, 4, 2]
foldbacks = [False, True, False, True, False]

print(f"\nPattern: {copy_numbers}")
print(f"Foldbacks: {foldbacks}")

generator = BFBCandidateGenerator(copy_numbers, foldbacks)
results = generator.run(max_cycles=3)
generator.print_summary(top_n=5)


DEMO 1: Basic Pattern

Pattern: [2, 4, 6, 4, 2]
Foldbacks: [False, True, False, True, False]
 BFBCandidateGenerator initialized
   Pattern: [2, 4, 6, 4, 2]
   Foldbacks: [False, True, False, True, False]
   Cycle types available: ['A', 'B', 'C']
BFB CANDIDATE GENERATOR

Input Pattern: [2, 4, 6, 4, 2]
Foldbacks: [False, True, False, True, False]
Searching up to 3 cycles...

Generated 39 possible sequences to evaluate

TOP 5 CANDIDATES

1. Sequence: A
   Match: 0.0% (0/4 positions)
   Predicted: [1, 1, 1, 1]
   Average Difference: 3.00

2. Sequence: B
   Match: 0.0% (0/3 positions)
   Predicted: [1, 1, 1]
   Average Difference: 3.00

3. Sequence: C
   Match: 0.0% (0/4 positions)
   Predicted: [1, 1, 1, 1]
   Average Difference: 3.00

4. Sequence: A -> A
   Match: 0.0% (0/5 positions)
   Predicted: [1, 1, 1, 1, 1, 1, 1, 1]
   Average Difference: 2.60

5. Sequence: A -> B
   Match: 0.0% (0/5 positions)
   Predicted: [1, 1, 1, 1, 1, 1]
   Average Difference: 2.60


In [5]:
# Demo 2 - Complex Pattern

print("\n" + "=" * 60)
print("DEMO 2: Complex Pattern")
print("=" * 60)

# Example 2: Complex pattern (longer sequence)
copy_numbers = [1, 2, 4, 8, 4, 2, 1]
foldbacks = [False, False, True, False, True, False, False]

print(f"\nPattern: {copy_numbers}")
print(f"Foldbacks: {foldbacks}")

generator = BFBCandidateGenerator(copy_numbers, foldbacks)
results = generator.run(max_cycles=4)
generator.print_summary(top_n=5)

# Export to file
generator.export_sequences("bfb_candidates_demo.txt")


DEMO 2: Complex Pattern

Pattern: [1, 2, 4, 8, 4, 2, 1]
Foldbacks: [False, False, True, False, True, False, False]
 BFBCandidateGenerator initialized
   Pattern: [1, 2, 4, 8, 4, 2, 1]
   Foldbacks: [False, False, True, False, True, False, False]
   Cycle types available: ['A', 'B', 'C']
BFB CANDIDATE GENERATOR

Input Pattern: [1, 2, 4, 8, 4, 2, 1]
Foldbacks: [False, False, True, False, True, False, False]
Searching up to 4 cycles...

Generated 120 possible sequences to evaluate

TOP 5 CANDIDATES

1. Sequence: B
   Match: 33.3% (1/3 positions)
   Predicted: [1, 1, 1]
   Average Difference: 1.33

2. Sequence: A -> A
   Match: 28.6% (2/7 positions)
   Predicted: [1, 1, 1, 1, 1, 1, 1, 1]
   Average Difference: 2.14

3. Sequence: A -> C
   Match: 28.6% (2/7 positions)
   Predicted: [1, 1, 1, 1, 1, 1, 1, 1]
   Average Difference: 2.14

4. Sequence: C -> A
   Match: 28.6% (2/7 positions)
   Predicted: [1, 1, 1, 1, 1, 1, 1, 1]
   Average Difference: 2.14

5. Sequence: C -> C
   Match: 28.6% (

In [6]:
# Demo 3 - Visual Demonstration (shows the generation process)

print("\n" + "=" * 60)
print("DEMO 3: Visual Demonstration")
print("=" * 60)

copy_numbers = [2, 4, 6]
foldbacks = [False, True, False]

print(f"\nObservation: {copy_numbers}")
print(f"Foldbacks: {foldbacks}")
print("\nGenerating all possible 2-cycle sequences...")
print("Showing how different sequences produce different patterns:")

generator = BFBCandidateGenerator(copy_numbers, foldbacks)

# Show how different sequences produce different patterns
sequences = generator.generate_sequences(max_cycles=2)

found_match = False
for seq in sequences:
    simulated = generator.simulate_bfb(seq)
    print(f"\n{seq} → {simulated}")
    if len(simulated) >= len(copy_numbers):
        # Check if first n elements match
        match = all(simulated[i] == copy_numbers[i] for i in range(len(copy_numbers)))
        if match:
            print("MATCH!")
            found_match = True

if not found_match:
    print("\nNo perfect match found. Try increasing max_cycles.")


DEMO 3: Visual Demonstration

Observation: [2, 4, 6]
Foldbacks: [False, True, False]

Generating all possible 2-cycle sequences...
Showing how different sequences produce different patterns:
 BFBCandidateGenerator initialized
   Pattern: [2, 4, 6]
   Foldbacks: [False, True, False]
   Cycle types available: ['A', 'B', 'C']

['A'] → [1, 1, 1, 1]

['B'] → [1, 1, 1]

['C'] → [1, 1, 1, 1]

['A', 'A'] → [1, 1, 1, 1, 1, 1, 1, 1]

['A', 'B'] → [1, 1, 1, 1, 1, 1]

['B', 'A'] → [1, 1, 1, 1, 1, 1]

['A', 'C'] → [1, 1, 1, 1, 1, 1, 1, 1]

['C', 'A'] → [1, 1, 1, 1, 1, 1, 1, 1]

['B', 'B'] → [1, 1, 1, 1, 1]

['B', 'C'] → [1, 1, 1, 1, 1, 1]

['C', 'B'] → [1, 1, 1, 1, 1, 1]

['C', 'C'] → [1, 1, 1, 1, 1, 1, 1, 1]

No perfect match found. Try increasing max_cycles.


In [7]:
# Demo 4 - Find All Matching Sequences

print("\n" + "=" * 60)
print("DEMO 4: Find All Matching Sequences")
print("=" * 60)

# Use a pattern that has multiple matches
copy_numbers = [2, 4, 6, 4, 2]
foldbacks = [False, True, False, True, False]

generator = BFBCandidateGenerator(copy_numbers, foldbacks)
results = generator.run(max_cycles=4)

print("\n" + "=" * 60)
print("ALL SEQUENCES WITH 100% MATCH")
print("=" * 60)

perfect_matches = [c for c in results if c['score'] >= 0.99]
print(f"\nFound {len(perfect_matches)} perfect matches:")

for i, match in enumerate(perfect_matches):
    sequence_str = " → ".join(match['sequence'])
    print(f"{i+1}. {sequence_str}")
    print(f"   Predicted: {match['predicted']}")


DEMO 4: Find All Matching Sequences
 BFBCandidateGenerator initialized
   Pattern: [2, 4, 6, 4, 2]
   Foldbacks: [False, True, False, True, False]
   Cycle types available: ['A', 'B', 'C']
BFB CANDIDATE GENERATOR

Input Pattern: [2, 4, 6, 4, 2]
Foldbacks: [False, True, False, True, False]
Searching up to 4 cycles...

Generated 120 possible sequences to evaluate

ALL SEQUENCES WITH 100% MATCH

Found 0 perfect matches:


In [8]:
# Interactive Demo - Try Your Own Pattern

print("\n" + "=" * 60)
print("DEMO 5: Try Your Own Pattern")
print("=" * 60)

# You can modify this cell with your own pattern
# Try different patterns to see how the generator works

# Example patterns to try:
# pattern_1 = [2, 4, 6, 4, 2]
# pattern_2 = [1, 2, 4, 8, 4, 2, 1]
# pattern_3 = [2, 4, 8, 4, 2]
# pattern_4 = [1, 2, 4, 6, 4, 2, 1]

# Choose your pattern here:
my_copy_numbers = [2, 4, 8, 4, 2]
my_foldbacks = [False, True, False, True, False]

print(f"\nYour Pattern: {my_copy_numbers}")
print(f"Your Foldbacks: {my_foldbacks}")

my_generator = BFBCandidateGenerator(my_copy_numbers, my_foldbacks)
my_results = my_generator.run(max_cycles=4)
my_generator.print_summary(top_n=5)

# Optional: Export your results
# my_generator.export_sequences("my_bfb_candidates.txt")


DEMO 5: Try Your Own Pattern

Your Pattern: [2, 4, 8, 4, 2]
Your Foldbacks: [False, True, False, True, False]
 BFBCandidateGenerator initialized
   Pattern: [2, 4, 8, 4, 2]
   Foldbacks: [False, True, False, True, False]
   Cycle types available: ['A', 'B', 'C']
BFB CANDIDATE GENERATOR

Input Pattern: [2, 4, 8, 4, 2]
Foldbacks: [False, True, False, True, False]
Searching up to 4 cycles...

Generated 120 possible sequences to evaluate

TOP 5 CANDIDATES

1. Sequence: A
   Match: 0.0% (0/4 positions)
   Predicted: [1, 1, 1, 1]
   Average Difference: 3.50

2. Sequence: B
   Match: 0.0% (0/3 positions)
   Predicted: [1, 1, 1]
   Average Difference: 3.67

3. Sequence: C
   Match: 0.0% (0/4 positions)
   Predicted: [1, 1, 1, 1]
   Average Difference: 3.50

4. Sequence: A -> A
   Match: 0.0% (0/5 positions)
   Predicted: [1, 1, 1, 1, 1, 1, 1, 1]
   Average Difference: 3.00

5. Sequence: A -> B
   Match: 0.0% (0/5 positions)
   Predicted: [1, 1, 1, 1, 1, 1]
   Average Difference: 3.00


In [9]:
# Quick Test - Verify Everything Works

print("\n" + "=" * 60)
print("QUICK TEST: Verifying All Functions")
print("=" * 60)

# Test 1: Can we create an instance?
test_generator = BFBCandidateGenerator([2, 4, 6], [False, True, False])
print("Test 1: Instance creation successful")

# Test 2: Can we simulate BFB?
test_sim = test_generator.simulate_bfb(['A', 'B'])
print(f"Test 2: Simulation works: {test_sim}")

# Test 3: Can we generate sequences?
test_seqs = test_generator.generate_sequences(max_cycles=2)
print(f"Test 3: Generated {len(test_seqs)} sequences")

# Test 4: Can we score?
test_score = test_generator.score_sequence(['A', 'B'])
print(f"Test 4: Scoring works: {test_score['match_percentage']:.1%}")

# Test 5: Can we run full pipeline?
test_results = test_generator.run(max_cycles=3)
print(f"Test 5: Full pipeline works, found {len(test_results)} candidates")

print("\n" + "=" * 60)
print("ALL TESTS PASSED!")
print("=" * 60)
print("\nBFB Candidate Generator is ready!")


QUICK TEST: Verifying All Functions
 BFBCandidateGenerator initialized
   Pattern: [2, 4, 6]
   Foldbacks: [False, True, False]
   Cycle types available: ['A', 'B', 'C']
Test 1: Instance creation successful
Test 2: Simulation works: [1, 1, 1, 1, 1, 1]
Test 3: Generated 12 sequences
Test 4: Scoring works: 0.0%
BFB CANDIDATE GENERATOR

Input Pattern: [2, 4, 6]
Foldbacks: [False, True, False]
Searching up to 3 cycles...

Generated 39 possible sequences to evaluate
Test 5: Full pipeline works, found 39 candidates

ALL TESTS PASSED!

BFB Candidate Generator is ready!


In [10]:
# Cell 10: Export Results

print("\n" + "=" * 60)
print("EXPORT RESULTS")
print("=" * 60)

# Use the generator from Demo 1 or 4
# This will create a text file you can download

copy_numbers = [2, 4, 6, 4, 2]
foldbacks = [False, True, False, True, False]

export_generator = BFBCandidateGenerator(copy_numbers, foldbacks)
export_generator.run(max_cycles=4)
export_generator.export_sequences("bfb_candidates_export.txt")

# print("\nFile saved as 'bfb_candidates_export.txt'")
# print("In Colab: File → Download → bfb_candidates_export.txt")


EXPORT RESULTS
 BFBCandidateGenerator initialized
   Pattern: [2, 4, 6, 4, 2]
   Foldbacks: [False, True, False, True, False]
   Cycle types available: ['A', 'B', 'C']
BFB CANDIDATE GENERATOR

Input Pattern: [2, 4, 6, 4, 2]
Foldbacks: [False, True, False, True, False]
Searching up to 4 cycles...

Generated 120 possible sequences to evaluate
Exported 120 candidates to bfb_candidates_export.txt (UTF-8)
